In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys

%cd /content
!wget https://github.com/propublica/compas-analysis/raw/master/compas-scores-two-years.csv
!if ! [ -d "Deep-SVDD-PyTorch" ]; then git clone https://github.com/nicbk/Deep-SVDD-PyTorch.git; fi
%cd Deep-SVDD-PyTorch
!git checkout -b compas
!git pull origin compas
%cd ..
!mkdir -p "Deep-SVDD-PyTorch-DATA"

sys.path.append('/content/Deep-SVDD-PyTorch/src')
%cd Deep-SVDD-PyTorch-DATA

/content
--2024-01-17 09:22:48--  https://github.com/propublica/compas-analysis/raw/master/compas-scores-two-years.csv
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv [following]
--2024-01-17 09:22:48--  https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2546489 (2.4M) [text/plain]
Saving to: ‘compas-scores-two-years.csv’

compas-scores-two-y 100%[===================>]   2.43M  --.-KB/s    in 0.06s   

2024-01-17 09:22:48 (38.4 MB/s) - ‘compas

In [3]:
settings = {
    'dataset_name': 'compas',
    'net_name': 'compas_Net',
    'xp_path': 'log/compas_high_score_nonrecidivate',
    'compas_path': '/content/compas-scores-two-years.csv',
    'train': True,
    'load_config': None,
    'load_model': None,
    'objective': 'one-class',
    'nu': 0.1,
    'device': 'cuda',
    'seed': -1,
    'optimizer_name': 'adam',
    'lr': 0.001,
    'n_epochs': 40,
    'lr_milestone': [30],
    'batch_size': 256,
    'weight_decay': 0.5e-6,
    'pretrain': True,
    'ae_optimizer_name': 'adam',
    'ae_lr': 0.001,
    'ae_n_epochs': 115,
    'ae_lr_milestone': [100],
    'ae_batch_size': 7500,
    'ae_weight_decay': 0.5e-6,
    'n_jobs_dataloader': 0,
    'quadrant': 1 # High decile score, did not reoffend
}

xp_path = settings['xp_path']
!mkdir -p $xp_path

In [ ]:
!cp -r -v /content/drive/MyDrive/CelebA/celeba_log/celeba_test_vae_smiling /content/Deep-SVDD-PyTorch-DATA/log/

In [4]:
import os
import click
import torch
import logging
import random
import numpy as np
import math
import matplotlib.pyplot as plt
import scipy
import copy

from PIL import Image
from utils.config import Config
from utils.visualization.plot_images_grid import plot_images_grid
from deepSVDD import DeepSVDD
from datasets.main import load_dataset

In [7]:
from numpy.core.numerictypes import obj2sctype

# Get configuration
cfg = Config(settings.copy())

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
log_file = cfg.settings['xp_path'] + '/log.txt'
file_handler = logging.FileHandler(log_file)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# If specified, load experiment config from JSON-file
if cfg.settings['load_config']:
    cfg.load_config(import_json=cfg.settings['load_config'])
    logger.info('Loaded configuration from %s.' % cfg.settings['load_config'])

# Print arguments
logger.info('Log file is %s.' % log_file)
logger.info('Export path is %s.' % cfg.settings['xp_path'])

logger.info('COMPAS Path: %s' % cfg.settings['compas_path'])
logger.info('Quadrant: %d' % cfg.settings['quadrant'])
logger.info('Network: %s' % cfg.settings['net_name'])

# Print configuration
logger.info('Deep SVDD objective: %s' % cfg.settings['objective'])
logger.info('Nu-parameter: %.2f' % cfg.settings['nu'])

# Set seed
if cfg.settings['seed'] != -1:
    random.seed(cfg.settings['seed'])
    np.random.seed(cfg.settings['seed'])
    torch.manual_seed(cfg.settings['seed'])
    logger.info('Set seed to %d.' % cfg.settings['seed'])

# Default device to 'cpu' if cuda is not available
if not torch.cuda.is_available():
    cfg.settings['device'] = 'cpu'
logger.info('Computation device: %s' % cfg.settings['device'])
logger.info('Number of dataloader workers: %d' % cfg.settings['n_jobs_dataloader'])

# Load data
dataset = load_dataset(cfg.settings['dataset_name'], cfg.settings['compas_path'], cfg.settings['quadrant'])

# Initialize DeepSVDD model and set neural network \phi
deep_SVDD = DeepSVDD(cfg.settings['objective'], cfg.settings['nu'], len(dataset.train_set[0][0]))
deep_SVDD.set_network(cfg.settings['net_name'])
# If specified, load Deep SVDD model (radius R, center c, network weights, and possibly autoencoder weights)
if cfg.settings['load_model']:
    deep_SVDD.load_model(model_path=cfg.settings['load_model'], load_ae=True)
    logger.info('Loading model from %s.' % cfg.settings['load_model'])

logger.info('Pretraining: %s' % cfg.settings['pretrain'])
if cfg.settings['pretrain'] and cfg.settings['train']:
    # Log pretraining details
    logger.info('Pretraining optimizer: %s' % cfg.settings['ae_optimizer_name'])
    logger.info('Pretraining learning rate: %g' % cfg.settings['ae_lr'])
    logger.info('Pretraining epochs: %d' % cfg.settings['ae_n_epochs'])
    logger.info('Pretraining learning rate scheduler milestones: %s' % (cfg.settings['ae_lr_milestone'],))
    logger.info('Pretraining batch size: %d' % cfg.settings['ae_batch_size'])
    logger.info('Pretraining weight decay: %g' % cfg.settings['ae_weight_decay'])

    # Pretrain model on dataset (via autoencoder)
    deep_SVDD.pretrain(dataset,
                       optimizer_name=cfg.settings['ae_optimizer_name'],
                       lr=cfg.settings['ae_lr'],
                       n_epochs=cfg.settings['ae_n_epochs'],
                       lr_milestones=cfg.settings['ae_lr_milestone'],
                       batch_size=cfg.settings['ae_batch_size'],
                       weight_decay=cfg.settings['ae_weight_decay'],
                       device=cfg.settings['device'],
                       n_jobs_dataloader=cfg.settings['n_jobs_dataloader'])

# Train model on dataset
if cfg.settings['train']:
    # Log training details
    logger.info('Training optimizer: %s' % cfg.settings['optimizer_name'])
    logger.info('Training learning rate: %g' % cfg.settings['lr'])
    logger.info('Training epochs: %d' % cfg.settings['n_epochs'])
    logger.info('Training learning rate scheduler milestones: %s' % (cfg.settings['lr_milestone'],))
    logger.info('Training batch size: %d' % cfg.settings['batch_size'])
    logger.info('Training weight decay: %g' % cfg.settings['weight_decay'])
    deep_SVDD.train(dataset,
                    optimizer_name=cfg.settings['optimizer_name'],
                    lr=cfg.settings['lr'],
                    n_epochs=cfg.settings['n_epochs'],
                    lr_milestones=cfg.settings['lr_milestone'],
                    batch_size=cfg.settings['batch_size'],
                    weight_decay=cfg.settings['weight_decay'],
                    device=cfg.settings['device'],
                    n_jobs_dataloader=cfg.settings['n_jobs_dataloader'])

# Test model
deep_SVDD.test(dataset, device=cfg.settings['device'], n_jobs_dataloader=cfg.settings['n_jobs_dataloader'])

# Plot most anomalous and most normal (within-class) test samples
init_idx = deep_SVDD.results['test_scores'][0][0]
index_scores = [(pair[0] - init_idx, pair[1]) for pair in deep_SVDD.results['test_scores']]
index_scores.sort(key=lambda x: x[1])
print(index_scores)
idx_sorted = [pair[0] for pair in index_scores]

if cfg.settings['dataset_name'] in ('compas'):
    if cfg.settings['dataset_name'] == 'compas':

        attributes = dataset.index_map

        ilp_tags_outliers = []

        #num_instances_outliers = math.floor(0.05 * len(idx_sorted))
        num_instances_outliers = 100
        for i in idx_sorted[-1:-(num_instances_outliers + 1):-1]:
            tags, _, _, _ = dataset.test_set[i]
            ilp_tags_outliers.append(tags.numpy())


        ilp_tags_normals = []

        #num_instances_normals = math.floor(0.05 * len(idx_sorted))
        num_instances_normals = 100
        for i in idx_sorted[:num_instances_normals]:
            tags, _, _, _ = dataset.test_set[i]
            ilp_tags_normals.append(tags.numpy())


# Save results, model, and configuration
if cfg.settings['train']:
    deep_SVDD.save_results(export_json=cfg.settings['xp_path'] + '/results.json')
    deep_SVDD.save_model(export_model=cfg.settings['xp_path'] + '/model.tar')
    cfg.save_config(export_json=cfg.settings['xp_path'] + '/config.json')

INFO:root:Log file is log/compas_high_score_nonrecidivate/log.txt.
INFO:root:Export path is log/compas_high_score_nonrecidivate.
INFO:root:COMPAS Path: /content/compas-scores-two-years.csv
INFO:root:Quadrant: 1
INFO:root:Network: compas_Net
INFO:root:Deep SVDD objective: one-class
INFO:root:Nu-parameter: 0.10
INFO:root:Computation device: cuda
INFO:root:Number of dataloader workers: 0
INFO:root:Pretraining: True
INFO:root:Pretraining optimizer: adam
INFO:root:Pretraining learning rate: 0.001
INFO:root:Pretraining epochs: 115
INFO:root:Pretraining learning rate scheduler milestones: [100]
INFO:root:Pretraining batch size: 7500
INFO:root:Pretraining weight decay: 5e-07
INFO:root:Starting pretraining...
INFO:root:  Epoch 1/115	 Time: 0.347	 Loss: 113.11292267
INFO:root:  Epoch 2/115	 Time: 0.341	 Loss: 112.58064270
INFO:root:  Epoch 3/115	 Time: 0.504	 Loss: 112.03905487
INFO:root:  Epoch 4/115	 Time: 0.334	 Loss: 111.48538971
INFO:root:  Epoch 5/115	 Time: 0.330	 Loss: 110.91645050
INFO:

[(1436, 2.2411110478515184e-07), (4, 2.241316394702153e-07), (18, 2.241316394702153e-07), (166, 2.241316394702153e-07), (235, 2.241316394702153e-07), (322, 2.241316394702153e-07), (770, 2.241316394702153e-07), (1279, 2.241316394702153e-07), (13, 3.538433475114289e-07), (91, 3.538433475114289e-07), (114, 3.538433475114289e-07), (174, 3.538433475114289e-07), (251, 3.538433475114289e-07), (308, 3.538433475114289e-07), (315, 3.538433475114289e-07), (354, 3.538433475114289e-07), (401, 3.538433475114289e-07), (412, 3.538433475114289e-07), (477, 3.538433475114289e-07), (493, 3.538433475114289e-07), (561, 3.538433475114289e-07), (565, 3.538433475114289e-07), (721, 3.538433475114289e-07), (761, 3.538433475114289e-07), (915, 3.538433475114289e-07), (988, 3.538433475114289e-07), (1016, 3.538433475114289e-07), (1192, 3.538433475114289e-07), (1270, 3.538433475114289e-07), (318, 3.6757697330358496e-07), (537, 3.6757697330358496e-07), (538, 3.6757697330358496e-07), (588, 3.6757697330358496e-07), (710

In [9]:
# Total number of edges in normal and outlier groups
num_tags = len(dataset.test_set[0][0])
num_edges_normals = sum([sum(instance) for instance in ilp_tags_normals[:num_instances_normals]])
num_edges_outliers = sum([sum(instance) for instance in ilp_tags_outliers[:num_instances_outliers]])

# Lists of tags per instance in normal and outlier groups
ilp_tags_normals_np = np.array(ilp_tags_normals)
ilp_tags_outliers_np = np.array(ilp_tags_outliers)

edge_sum_normals = [(sum(ilp_tags_normals_np[:, i])/num_edges_normals
                   - sum(ilp_tags_outliers_np[:, i])/num_edges_outliers,
                     attributes[i]) for i in range(num_tags)]
edge_sum_outliers = [(sum(ilp_tags_outliers_np[:, i])/num_edges_outliers
                    - sum(ilp_tags_normals_np[:, i])/num_edges_normals,
                      attributes[i]) for i in range(num_tags)]

edge_sum_normals.sort(reverse=True, key = lambda x: x[0])
edge_sum_outliers.sort(reverse=True, key = lambda x: x[0])

simple_normals = set()
simple_outliers = set()

print('')
print('Normal Explanation:')
print('')
for i in range(54):
    if edge_sum_normals[i][0] <= 0:
        break
    if edge_sum_normals[i][1] != attributes[settings['quadrant']]:
        tabs = '\t\t\t'
        tag = edge_sum_normals[i][1]
        if len(tag) < 8:
          tabs += '\t\t'
        elif len(tag) < 16:
          tabs += '\t'
        print(edge_sum_normals[i][1] + tabs + 'Differential Edge Coverage: ' + str(edge_sum_normals[i][0] * 100) + '%')
        simple_normals.add(edge_sum_normals[i][1])

print('')
print('Outlier Explanation:')
print('')
for i in range(54):
    if edge_sum_outliers[i][0] <= 0:
        break
    if edge_sum_outliers[i][1] != attributes[settings['quadrant']]:
        tabs = '\t\t\t'
        tag = edge_sum_outliers[i][1]
        if len(tag) < 8:
          tabs += '\t\t'
        elif len(tag) < 16:
          tabs += '\t'
        print(edge_sum_outliers[i][1] + tabs + 'Differential Edge Coverage: ' + str(edge_sum_outliers[i][0] * 100) + '%')
        simple_outliers.add(edge_sum_outliers[i][1])


Normal Explanation:

arrest case no charge			Differential Edge Coverage: 9.2%
Battery					Differential Edge Coverage: 4.2%
25 - 45					Differential Edge Coverage: 2.5999999999999996%
African-American			Differential Edge Coverage: 1.8000000000000003%
Susp Drivers Lic 1st Offense			Differential Edge Coverage: 1.7999999999999998%
Female					Differential Edge Coverage: 1.6%
Possession of Cocaine			Differential Edge Coverage: 1.4000000000000001%
Grand Theft in the 3rd Degree			Differential Edge Coverage: 1.4000000000000001%
Possession Burglary Tools			Differential Edge Coverage: 0.8%
Caucasian				Differential Edge Coverage: 0.5999999999999992%
Driving While License Revoked			Differential Edge Coverage: 0.4%
M					Differential Edge Coverage: 0.3999999999999997%
Burglary Dwelling Assault/Batt			Differential Edge Coverage: 0.2%
Poss Pyrrolidinovalerophenone			Differential Edge Coverage: 0.2%
Resist Officer w/Violence			Differential Edge Coverage: 0.2%

Outlier Explanation:

Male					Different